In [ ]:
# ============================================================
# 1. Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# 2. Import Libraries
# ============================================================

import pandas as pd
import os
from sklearn.model_selection import train_test_split

In [ ]:
# ============================================================
# 3. Load Excel Dataset
# ============================================================

file_path = "/content/drive/MyDrive/Thesis/FineTunning/cancercolumnsdata.xlsx"

df = pd.read_excel(file_path)

print("Total records:", len(df))
print("Columns:")
print(df.columns)

Total records: 9439
Columns:
Index(['Age', 'Sex', 'Marital status at diagnosis', 'Primary Site - labeled',
       'Grade', 'T stage', 'N stage', 'M stage', 'Radiotherapy',
       'Surgical treatment', 'Chemotherapy', 'Survival months',
       'Overall survival'],
      dtype='object')


In [ ]:
# ============================================================
# 4. Radiotherapy Column Mapping
# ============================================================

# Change this only if your exact column name is different
radiotherapy_col = "Radiotherapy"

df[radiotherapy_col] = df[radiotherapy_col].astype("string").str.strip()

radiotherapy_map = {
    "No radiation": "Non-radiotherapy",
    "Radiotherapy": "Radiotherapy",
    "Radiation prior to surgery": "Radiotherapy",
    "Intraoperative radiation": "Radiotherapy",
    "Sequence unknown, but both were given": "Radiotherapy",
    "Radiation before and after surgery": "Radiotherapy",
    "Intraoperative rad with other rad before/after surgery": "Radiotherapy",
    "Surgery both before and after radiation": "Radiotherapy"
}

df["radiotherapy_label"] = df[radiotherapy_col].map(radiotherapy_map)

# Check unmapped values
unmapped = df[df["radiotherapy_label"].isna()][radiotherapy_col].unique()

print("Unmapped values:")
print(unmapped)

# Numeric label for ML
df["radiotherapy_binary"] = df["radiotherapy_label"].map({
    "Non-radiotherapy": 0,
    "Radiotherapy": 1
})

print(df["radiotherapy_label"].value_counts())

Unmapped values:
<StringArray>
[]
Length: 0, dtype: string
radiotherapy_label
Non-radiotherapy    7325
Radiotherapy        2114
Name: count, dtype: int64


In [ ]:
# ============================================================
# 5. Split Dataset into Exact Portions
# ============================================================

TRAIN_SIZE = 7551
TEST_SIZE = 944
VAL_SIZE = 944

total_expected = TRAIN_SIZE + TEST_SIZE + VAL_SIZE

assert len(df) == total_expected, f"Expected {total_expected} records, but found {len(df)}"

# First split: Train = 7551, Remaining = 1888
train_df, remaining_df = train_test_split(
    df,
    train_size=TRAIN_SIZE,
    random_state=42,
    shuffle=True,
    stratify=df["radiotherapy_label"]
)

# Second split: Test = 944, Validation = 944
test_df, validation_df = train_test_split(
    remaining_df,
    train_size=TEST_SIZE,
    test_size=VAL_SIZE,
    random_state=42,
    shuffle=True,
    stratify=remaining_df["radiotherapy_label"]
)

print("Train records:", len(train_df))
print("Test records:", len(test_df))
print("Validation records:", len(validation_df))

Train records: 7551
Test records: 944
Validation records: 944


In [ ]:
# ============================================================
# 6. Check Distribution
# ============================================================

print("Training distribution:")
print(train_df["radiotherapy_label"].value_counts())

print("\nTesting distribution:")
print(test_df["radiotherapy_label"].value_counts())

print("\nValidation distribution:")
print(validation_df["radiotherapy_label"].value_counts())

Training distribution:
radiotherapy_label
Non-radiotherapy    5860
Radiotherapy        1691
Name: count, dtype: int64

Testing distribution:
radiotherapy_label
Non-radiotherapy    732
Radiotherapy        212
Name: count, dtype: int64

Validation distribution:
radiotherapy_label
Non-radiotherapy    733
Radiotherapy        211
Name: count, dtype: int64


In [ ]:
# ============================================================
# 7. Save Split Files Back to Google Drive
# ============================================================

output_dir = "/content/drive/MyDrive/Thesis/FineTunning/split_dataset"

os.makedirs(output_dir, exist_ok=True)

train_df.to_csv(f"{output_dir}/train.csv", index=False)
test_df.to_csv(f"{output_dir}/test.csv", index=False)
validation_df.to_csv(f"{output_dir}/validation.csv", index=False)

train_df.to_excel(f"{output_dir}/train.xlsx", index=False)
test_df.to_excel(f"{output_dir}/test.xlsx", index=False)
validation_df.to_excel(f"{output_dir}/validation.xlsx", index=False)

print("Files saved successfully in:")
print(output_dir)

Files saved successfully in:
/content/drive/MyDrive/Thesis/FineTunning/split_dataset


In [ ]:
print(df.columns.tolist())

['Age', 'Sex', 'Marital status at diagnosis', 'Primary Site - labeled', 'Grade', 'T stage', 'N stage', 'M stage', 'Radiotherapy', 'Surgical treatment', 'Chemotherapy', 'Survival months', 'Overall survival', 'radiotherapy_label', 'radiotherapy_binary']
